In [1]:
import os
import json
import yaml
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from umap.umap_ import UMAP
import seaborn as sns
from huggingface_hub import login
from datasets import load_dataset, Dataset
from sklearn.metrics import cohen_kappa_score


In [2]:
def load_personas(hf_persona_path,n_personasample):
    
    print("Using Huggingface Personas")
    print(f"Loading Personas from {hf_persona_path}")
    hf_persona_dataset = load_dataset(hf_persona_path)
    persona_datasets_total = hf_persona_dataset['train']
    total_persona_df = persona_datasets_total.to_pandas()
    persona_datasets = total_persona_df.groupby("archetype").sample(n=n_personasample, random_state=42)
    print(f"No of Personas: {persona_datasets.shape[0]}")
    return persona_datasets

def load_sjts(hf_sjt_path,n_sjtsample):
    
    print("Using Huggingface SJTs")
    print(f"Loading SJTs from {hf_sjt_path}")
    hf_sjt_dataset = load_dataset(hf_sjt_path)
    sjt_datasets_total = hf_sjt_dataset['train']
    total_sjt_df = sjt_datasets_total.to_pandas()
    sjt_datasets = total_sjt_df.groupby("template_no").sample(n=n_sjtsample, random_state=42)
    print(f"No of SJTs: {sjt_datasets.shape[0]}")
    return sjt_datasets

In [3]:
trait_map = {
    "1": "Honesty-Humility",
    "2": "Emotionality",
    "3": "Extraversion",
    "4": "Agreeableness",
    "5": "Conscientiousness",
    "6": "Openness"
}

In [4]:
def write_to_json(file, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w') as f:
        json.dump(file, f, indent=2)


def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)
    
# def load_sjt(sjt_dir):
#     sjt_list = []
#     for file in os.listdir(sjt_dir):
#         synthetic_sjt = read_json(os.path.join(sjt_dir, file))
#         sjt_list += synthetic_sjt

#     print(f"{len(sjt_list)} SJTs Loaded")
#     return sjt_list

def get_model_name(filename, file_str):
    model_name = filename.replace("_sjt_answers_","").replace(".json","").replace(file_str,"")
    return model_name

def normalize_sjt_responses(chosen_indices, n_options=6):
    """
    Returns an (n_questions x n_options) matrix where chosen = +1, others = -1.
    chosen_indices is a list or array of 1-indexed chosen options.
    """
    import numpy as np
    chosen_indices = np.array(chosen_indices)
    norm_matrix = -1 * np.ones((len(chosen_indices), n_options), dtype=int)
    norm_matrix[np.arange(len(chosen_indices)), chosen_indices - 1] = 1
    return norm_matrix

In [5]:
def get_true_indices_v1(data):
    resolved_traits = []
    true_answer_indices = []
    for answer, shuffled_indices in zip(data["answers"][0], data['config']['answer_index'][0]):
        # assuming one list per persona
        # shuffled_indices = data["config"]["answer_index"]  # list of per-question shuffles

        # Map answers back to their intended trait meaning
        # resolved_traits = []
        # true_answer_indices = []
        # for ans, shuffle in zip(answer, shuffled_indices):
            # ans is the chosen index (1,2,3,...)
        # shuffle is the shuffled order of indices for this question
        true_index = str(shuffled_indices[int(answer)-1] + 1)  # get original index meaning
        if true_index in trait_map:
            resolved_traits.append(trait_map[true_index])
            true_answer_indices.append(true_index)
    
    return true_answer_indices, resolved_traits

def get_true_indices(data):
    resolved_traits = []
    true_answer_indices = []
    for answer, shuffled_indices in zip(data["answers"][0], data['config']['answer_index']):
        # assuming one list per persona
        # shuffled_indices = data["config"]["answer_index"]  # list of per-question shuffles

        # Map answers back to their intended trait meaning
        # resolved_traits = []
        # true_answer_indices = []
        # for ans, shuffle in zip(answer, shuffled_indices):
            # ans is the chosen index (1,2,3,...)
        # shuffle is the shuffled order of indices for this question
        true_index = str(shuffled_indices[int(answer)-1] + 1)  # get original index meaning
        if true_index in trait_map:
            resolved_traits.append(trait_map[true_index])
            true_answer_indices.append(true_index)
    
    return true_answer_indices, resolved_traits

In [6]:
zero_compute_data_dir = "../experiment_results/zero_compute_analysis_sjt/"

In [ ]:
file_str = "huggingface"
# hf_sjt_trait_distributions_df = pd.DataFrame()
for filename in os.listdir(zero_compute_data_dir):
    if ".json" in filename:
        print(filename)
        
        model_name = get_model_name(filename, file_str)
        hf_persona_sjt_results = read_json(os.path.join(zero_compute_data_dir, filename))
        
        persona_sjt_str_answers_df = pd.DataFrame([{"persona_id": key, "answers": get_true_indices(hf_persona_sjt_results[key])[0]} for key in hf_persona_sjt_results.keys()])
        # persona_sjt_str_answers_df = pd.DataFrame([{"persona_id": key, "answers": get_trait_distribution(hf_persona_sjt_results[key])} for key in hf_persona_sjt_results.keys()])
        persona_sjt_str_answers_df = pd.DataFrame(persona_sjt_str_answers_df['answers'].to_list(), columns = list(range(0,500)), index = persona_sjt_str_answers_df['persona_id'])
        # persona_sjt_str_answers_df['model_name'] = model_name



In [6]:
for filename in os.listdir(zero_compute_data_dir):
    if ".json" in filename:
        print(filename)

huggingface_sjt_answers_gpt-4_1.json
huggingface_sjt_answers_Qwen2_5-7B-Instruct.json
huggingface_sjt_answers_Llama-3_1-8B-Instruct.json


In [7]:
gpt_hf_persona_sjt_results = read_json(os.path.join(zero_compute_data_dir, "huggingface_sjt_answers_gpt-4_1.json"))
llama_hf_persona_sjt_results = read_json(os.path.join(zero_compute_data_dir, "huggingface_sjt_answers_Llama-3_1-8B-Instruct.json"))
qwen_hf_persona_sjt_results = read_json(os.path.join(zero_compute_data_dir, "huggingface_sjt_answers_Qwen2_5-7B-Instruct.json"))

In [37]:
def json_to_df_sjt(input_json, index=0):
    input_answers_df = pd.DataFrame([{"persona_id": key, "answers": get_true_indices(input_json[key])[index]} for key in input_json.keys()])

    input_answers_df = pd.DataFrame(input_answers_df['answers'].to_list(), columns = list(range(0,500)), index = input_answers_df['persona_id'])    
    return input_answers_df

In [15]:
gpt_sjt_df = json_to_df_sjt(gpt_hf_persona_sjt_results)
llama_sjt_df = json_to_df_sjt(llama_hf_persona_sjt_results)
qwen_sjt_df = json_to_df_sjt(qwen_hf_persona_sjt_results)

In [16]:
gpt_sjt_df.shape, llama_sjt_df.shape, qwen_sjt_df.shape

((200, 500), (200, 500), (200, 500))

In [27]:
agreement_list = []
for persona_id in list(gpt_sjt_df.index):
    
    gpt_answers = list(gpt_sjt_df.loc[persona_id])
    llama_answers = list(llama_sjt_df.loc[persona_id])
    qwen_answers = list(qwen_sjt_df.loc[persona_id])
    
    gpt_vs_llama = cohen_kappa_score(gpt_answers, llama_answers) 
    gpt_vs_qwen = cohen_kappa_score(gpt_answers, qwen_answers)
    llama_vs_qwen = cohen_kappa_score(llama_answers, qwen_answers)
    
    final_dict = {
        "persona_id" : persona_id,
        "gpt_vs_llama": gpt_vs_llama,
        "gpt_vs_qwen" : gpt_vs_qwen,
        "llama_vs_qwen": llama_vs_qwen
    }
    
    agreement_list.append(final_dict)

In [30]:
pd.DataFrame(agreement_list).describe()

,gpt_vs_llama,gpt_vs_qwen,llama_vs_qwen
count,200.000000,200.000000,200.000000
mean,0.254500,0.271237,0.273954
std,0.055046,0.069507,0.042959
min,0.095460,0.061341,0.168360
25%,0.219350,0.231491,0.249219
50%,0.259690,0.286532,0.276198
75%,0.296805,0.327860,0.300995
max,0.401363,0.402390,0.386486


In [38]:
gpt_sjt_trait_df = json_to_df_sjt(gpt_hf_persona_sjt_results, index =1)
llama_sjt_trait_df = json_to_df_sjt(llama_hf_persona_sjt_results, index=1)
qwen_sjt_trait_df = json_to_df_sjt(qwen_hf_persona_sjt_results, index=1)

In [44]:
def get_trait_distribution(df):
    
    # Step 1: get all unique values
    unique_vals = pd.unique(df.values.ravel())

    # Step 2: create an empty output dataframe
    out = pd.DataFrame(0, index=df.index, columns=unique_vals, dtype=float)

    # Step 3: count occurrences of each value across rows (fully vectorized)
    for val in unique_vals:
        out[val] = (df == val).sum(axis=1)

    # Step 4: convert counts to proportions
    out = out.div(df.shape[1])
    
    return out

In [56]:
trait_distribution_df = pd.concat([get_trait_distribution(gpt_sjt_trait_df).mean(), get_trait_distribution(llama_sjt_trait_df).mean(), get_trait_distribution(qwen_sjt_trait_df).mean()],axis = 1)

trait_distribution_df.columns = ['gpt','llama','qwen']

In [57]:
trait_distribution_df

,gpt,llama,qwen
Agreeableness,0.10773,0.12970,0.16265
Honesty-Humility,0.41990,0.52032,0.40122
Conscientiousness,0.36301,0.18605,0.20380
Emotionality,0.02084,0.03887,0.02949
Openness,0.04632,0.07387,0.15226
Extraversion,0.04220,0.05119,0.05058


In [31]:
zero_compute_hexaco_dir = "../experiment_results/zero_compute_analysis_hexaco/"

gpt_hf_persona_hexaco_results = read_json(os.path.join(zero_compute_hexaco_dir, "huggingface_hexaco_answers_gpt-4_1.json"))
llama_hf_persona_hexaco_results = read_json(os.path.join(zero_compute_hexaco_dir, "huggingface_hexaco_answers_Llama-3_1-8B-Instruct.json"))
qwen_hf_persona_hexaco_results = read_json(os.path.join(zero_compute_hexaco_dir, "huggingface_hexaco_answers_Qwen2_5-7B-Instruct.json"))

In [32]:
def json_to_df_hexaco(input_dict):
    input_answers_df = pd.DataFrame([{"persona_id": key, "answers": input_dict[key]['answers'][0]} for key in input_dict.keys()])
    input_answers_df = pd.DataFrame(input_answers_df['answers'].to_list(), columns = list(range(0,100)), index = input_answers_df['persona_id'])
    
    return input_answers_df

In [33]:
gpt_hexaco_df = json_to_df_hexaco(gpt_hf_persona_hexaco_results)
llama_hexaco_df = json_to_df_hexaco(llama_hf_persona_hexaco_results)
qwen_hexaco_df = json_to_df_hexaco(qwen_hf_persona_hexaco_results)

In [34]:
gpt_hexaco_df.shape, llama_hexaco_df.shape, qwen_hexaco_df.shape

((200, 100), (200, 100), (200, 100))

In [35]:
hexaco_agreement_list = []
for persona_id in list(gpt_hexaco_df.index):
    
    gpt_answers = list(gpt_hexaco_df.loc[persona_id])
    llama_answers = list(llama_hexaco_df.loc[persona_id])
    qwen_answers = list(qwen_hexaco_df.loc[persona_id])
    
    gpt_vs_llama = cohen_kappa_score(gpt_answers, llama_answers) 
    gpt_vs_qwen = cohen_kappa_score(gpt_answers, qwen_answers)
    llama_vs_qwen = cohen_kappa_score(llama_answers, qwen_answers)
    
    final_dict = {
        "persona_id" : persona_id,
        "gpt_vs_llama": gpt_vs_llama,
        "gpt_vs_qwen" : gpt_vs_qwen,
        "llama_vs_qwen": llama_vs_qwen
    }
    
    hexaco_agreement_list.append(final_dict)

In [36]:
pd.DataFrame(hexaco_agreement_list).describe()

,gpt_vs_llama,gpt_vs_qwen,llama_vs_qwen
count,200.000000,200.000000,200.000000
mean,0.235313,0.124019,0.151534
std,0.067977,0.058001,0.065110
min,0.091649,-0.037471,-0.016382
25%,0.184473,0.090535,0.111594
50%,0.234019,0.118969,0.150000
75%,0.274309,0.161754,0.195803
max,0.414954,0.287207,0.319641
